# C3 · Dynamics: how loud, and how it changes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/C3_dynamics.ipynb)

*Dynamics* is how loud the music is, and how the level changes over time. This chapter computes two features from MIRtoolbox:

- the **RMS energy**, a measure of how strong the sound wave is; and
- the **low-energy rate**, which tells whether the level stays even or comes in bursts.

RMS is a physical measure. It follows how loud we *hear* a sound only roughly: our ears are more sensitive to some frequencies than others.

Each section has the same steps: ① the idea, with sound; ② the maths in NumPy; ③ mirpy; ④ librosa; ⑤ Essentia; ⑥ the three libraries side by side; and an exercise.

## Setup

In Colab, the next cell installs the libraries. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q git+https://github.com/juansgomez87/py-mir-toolbox librosa essentia"
    )

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio

import mirpy as mir

try:
    import essentia.standard as es
except ImportError:
    es = None
    print("Essentia is not installed here; the Essentia cells will be skipped.")

We use two seconds of synthetic sound. A **crescendo** is a tone that starts very soft and ends loud. The player does not adjust the volume, so you hear the real change in level.

In [ ]:
sr = 22050  # samples per second
t = np.arange(2 * sr) / sr  # two seconds
envelope = np.linspace(0.01, 1, t.size)  # the loudness shape: soft to loud
crescendo = envelope * np.sin(2 * np.pi * 220 * t)

Audio(crescendo, rate=sr, normalize=False)

## 1. Frames

### ① Idea

The level of music changes all the time, so one number for the whole piece is not enough. We cut the sound into short, overlapping pieces called **frames** and compute the feature on each one. The result is a curve that follows the music over time.

MIRtoolbox uses frames of 50 ms. Each frame starts 25 ms after the previous one: this step is called the **hop**. So each frame overlaps half of the next.

### ② From scratch

At 22 050 samples per second, 50 ms is 1102 samples (1102.5, rounded down) and the hop is 551.25 samples. MIRtoolbox keeps the hop as a fraction and rounds each frame's start down to a whole sample. It keeps only whole frames, so the very end of the sound, too short for a frame, is left out.

In [ ]:
frame_seconds = 0.05
hop_ratio = 0.5  # hop = half a frame

frame_length = int(frame_seconds * sr)  # samples per frame
hop = hop_ratio * frame_seconds * sr  # samples between frame starts
n_frames = int((crescendo.size - frame_length) // hop) + 1
starts = np.floor(np.arange(n_frames) * hop).astype(int)

print(f"{frame_length} samples per frame, hop {hop} samples, {n_frames} frames")
print("first frame starts:", starts[:5])

The plot shows the first five frames over the waveform. Each bar is one frame; they overlap by half.

In [ ]:
def draw_frames(ax, frame_starts, length, n_shown=5, label=None):
    """Draw the first frames as horizontal bars, one row per frame."""
    for i, start in enumerate(frame_starts[:n_shown]):
        ax.barh(-0.2 - 0.15 * i, length, left=start, height=0.1, color=f"C{i}", alpha=0.7)
    if label:
        ax.text(0.99, 0.95, label, transform=ax.transAxes, ha="right", va="top")


shown = slice(0, int(0.15 * sr))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t[shown], crescendo[shown], color="gray", linewidth=0.8)
draw_frames(ax, starts / sr, frame_length / sr)
ax.set_xlabel("Time (s)")
ax.set_yticks([])
ax.set_title("Waveform and the first five frames")
plt.show()

### ③ mirpy

`mir.frame` cuts a sound into frames. You give the frame length in seconds and the hop as a share of the frame, as in MATLAB `mirframe(a, 0.05, 0.5)`. All mirpy features take the same setting through their `frame` argument.

The result keeps the frames side by side: its shape is (channels, samples per frame, frames).

In [ ]:
a = mir.audio(crescendo, sr=sr)
framed = mir.frame(a, (frame_seconds, hop_ratio))

print(framed.data.shape)
print("same frame starts as our NumPy version:", np.allclose(framed.frame_starts * sr, starts))

### ④ librosa

`librosa.util.frame` also cuts a signal into frames. Two things work differently:

- the hop is a whole number of samples (551 instead of 551.25);
- librosa's features usually **centre** the frames: they add half a frame of silence before the start, so that the first frame is centred at time 0 instead of starting there.

`librosa.util.frame` itself does not centre. The features do it for you, with `center=True` by default.

In [ ]:
librosa_frames = librosa.util.frame(crescendo, frame_length=frame_length, hop_length=int(hop))
print(librosa_frames.shape)  # (samples per frame, frames)

librosa_starts = np.arange(librosa_frames.shape[1]) * int(hop)
centred_starts = librosa_starts - frame_length // 2  # where centred frames start

### ⑤ Essentia

Essentia's `FrameGenerator` gives the frames one by one, so you can compute a feature on each. It also uses a whole-number hop. By default it centres the first frame on time 0, like librosa; `startFromZero=True` starts it at 0, like MIRtoolbox. Unlike the others, it also keeps a last frame at the end of the sound, filled up with silence.

In [ ]:
if es is None:
    print("Essentia is not available; skipped.")
else:
    essentia_frames = list(
        es.FrameGenerator(
            crescendo.astype(np.float32),
            frameSize=frame_length,
            hopSize=int(hop),
            startFromZero=True,
        )
    )
    print(len(essentia_frames), "frames of", essentia_frames[0].size, "samples")

### ⑥ Side by side

The first frames of each library. mirpy and Essentia (with `startFromZero=True`) start at 0; librosa's centred frames start half a frame earlier.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
layouts = [
    ("mirpy", starts / sr),
    ("librosa (centred)", centred_starts / sr),
    ("Essentia (startFromZero)", librosa_starts / sr),
]
for ax, (name, frame_starts) in zip(axes, layouts, strict=True):
    ax.plot(t[shown], crescendo[shown], color="gray", linewidth=0.8)
    draw_frames(ax, frame_starts, frame_length / sr, label=name)
    ax.set_yticks([])
axes[-1].set_xlabel("Time (s)")
axes[0].set_xlim(-0.03, 0.15)
plt.show()

### Exercise

With 100 ms frames and a hop of a quarter of a frame, how many frames does the crescendo have? Work it out with the formula used in ②, then check with `mir.frame`.

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
length_100 = int(0.1 * sr)
hop_100 = 0.25 * 0.1 * sr
print(int((crescendo.size - length_100) // hop_100) + 1)
print(mir.frame(a, (0.1, 0.25)).n_frames)

## 2. RMS energy

### ① Idea

A sound is louder when its wave swings further from zero. The **root-mean-square** (RMS) measures this in three steps:

1. **square** every sample, so that swings up and down both count as positive;
2. take the **mean** of the squares;
3. take the square **root**, to come back to the scale of the wave.

$$\mathrm{RMS} = \sqrt{\frac{1}{N} \sum_{n=1}^{N} x_n^2}$$

Computed on every frame, RMS gives a curve of the level over time. For the crescendo, it should rise from soft to loud.

### ② From scratch

The three steps on one frame from the middle of the crescendo:

In [ ]:
middle = crescendo[starts[40] : starts[40] + frame_length]
squares = middle**2
rms_value = np.sqrt(np.mean(squares))

frame_time = np.arange(frame_length) / sr * 1000  # in ms
fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
axes[0].plot(frame_time, middle)
axes[0].set_title("one frame")
axes[1].plot(frame_time, squares, color="C1")
axes[1].axhline(np.mean(squares), color="k", linestyle="--")
axes[1].set_title("1. square, 2. mean (dashed)")
axes[2].plot(frame_time, middle, color="lightgray")
axes[2].axhline(rms_value, color="C2", linewidth=2)
axes[2].set_title("3. square root = RMS (green)")
for ax in axes:
    ax.set_xlabel("Time in the frame (ms)")
plt.show()

The same on every frame gives the RMS curve:

In [ ]:
frames = crescendo[starts[:, np.newaxis] + np.arange(frame_length)]  # one row per frame
rms_curve = np.sqrt(np.mean(frames**2, axis=1))
frame_centres = (starts + frame_length / 2) / sr

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, crescendo, color="lightgray", linewidth=0.5, label="waveform")
ax.plot(frame_centres, rms_curve, label="RMS of each frame")
ax.set_xlabel("Time (s)")
ax.legend()
plt.show()

### ③ mirpy

`mir.rms` is MIRtoolbox `mirrms`. Without `frame` it gives one value for the whole sound; with `frame=True` it uses 50 ms frames and a hop of half a frame. Every mirpy result has a `.plot()` method.

(For a whole sound, MIRtoolbox divides by one sample less than the formula above, $N - 1$ instead of $N$. mirpy does the same, so that results match MIRtoolbox; see the [Parity report](../parity.md).)

In [ ]:
print(f"RMS of the whole crescendo: {mir.rms(a).data.item():.3f}")

curve = mir.rms(a, frame=True)
curve.plot()
plt.show()

### ④ librosa

`librosa.feature.rms` takes the frame length and hop in samples. It centres the frames by default (see section 1). `librosa.times_like` gives the time of each frame.

In [ ]:
librosa_rms = librosa.feature.rms(y=crescendo, frame_length=frame_length, hop_length=int(hop))[0]
librosa_times = librosa.times_like(librosa_rms, sr=sr, hop_length=int(hop))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(librosa_times, librosa_rms)
ax.set_xlabel("Time (s)")
ax.set_title("librosa.feature.rms")
plt.show()

The curve starts lower than mirpy's: the first centred frame is half silence (the padding added before the sound).

### ⑤ Essentia

In Essentia, the frames come from `FrameGenerator` and the `RMS` algorithm is applied to each one.

In [ ]:
def essentia_rms_curve(x):
    """RMS of each frame with Essentia, the first frame starting at time 0."""
    frames = es.FrameGenerator(
        x.astype(np.float32), frameSize=frame_length, hopSize=int(hop), startFromZero=True
    )
    rms_algorithm = es.RMS()
    return np.array([rms_algorithm(frame) for frame in frames])


if es is None:
    print("Essentia is not available; skipped.")
else:
    essentia_rms = essentia_rms_curve(crescendo)
    essentia_times = (np.arange(essentia_rms.size) * int(hop) + frame_length / 2) / sr

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(essentia_times, essentia_rms)
    ax.set_xlabel("Time (s)")
    ax.set_title("Essentia RMS")
    plt.show()

The last value drops: it is the extra frame at the end, partly filled with silence.

### ⑥ Side by side

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(curve.frame_centers, curve.data[0, 0], linewidth=3, alpha=0.5, label="mirpy")
ax.plot(librosa_times, librosa_rms, label="librosa")
if es is not None:
    ax.plot(essentia_times, essentia_rms, linestyle="--", label="Essentia")
ax.set_xlabel("Time (s)")
ax.set_ylabel("RMS")
ax.legend()
plt.show()

The three curves lie on top of each other. They only differ at the edges, where each library treats the start and end of the sound in its own way:

- **mirpy** follows MIRtoolbox: frames start at 0, and the end that does not fill a frame is left out.
- **librosa** centres the frames, so its curve starts with a half-silent frame.
- **Essentia** adds a last frame filled up with silence.

### Exercise

We hear loudness on a roughly logarithmic scale, measured in **decibels** (dB): $20 \log_{10}(\mathrm{RMS})$.

1. Plot the RMS curve of the crescendo in dB. When does the level change fastest? Does that match what you hear?
2. Build a crescendo whose envelope goes from −40 dB to 0 dB in equal steps of dB, listen to it, and plot its RMS in dB.

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
db_envelope = 10 ** (np.linspace(-40, 0, t.size) / 20)
db_crescendo = db_envelope * np.sin(2 * np.pi * 220 * t)
db_curve = mir.rms(mir.audio(db_crescendo, sr=sr), frame=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(curve.frame_centers, 20 * np.log10(curve.data[0, 0]), label="first crescendo")
ax.plot(db_curve.frame_centers, 20 * np.log10(db_curve.data[0, 0]), label="crescendo in dB steps")
ax.set_xlabel("Time (s)")
ax.set_ylabel("RMS (dB)")
ax.legend()
plt.show()

Audio(db_crescendo, rate=sr, normalize=False)

The first crescendo rises fastest in dB at the start and then flattens. The second rises at a steady 20 dB per second and sounds more even.

## 3. Low-energy rate

### ① Idea

The **low-energy rate** is the share of frames that are quieter than the average frame (Tzanetakis and Cook, 2002).

- A sound with an even level has about half its frames below the average: a rate near **0.5**.
- A sound made of a few loud hits with quiet gaps has a high rate: the hits raise the average, and most frames fall below it.

Two examples: a held tone that swells slightly, and a series of short drum-like hits.

In [ ]:
sustained = (0.5 + 0.1 * np.sin(2 * np.pi * 2 * t)) * np.sin(2 * np.pi * 220 * t)

rng = np.random.default_rng(0)
hit_times = np.arange(0, 2, 0.25)
decay = np.exp(-(t[:, np.newaxis] - hit_times) / 0.02) * (t[:, np.newaxis] >= hit_times)
percussive = decay.sum(axis=1) * rng.standard_normal(t.size)
percussive *= 0.9 / np.max(np.abs(percussive))  # loudest sample at 0.9, no clipping

Audio(sustained, rate=sr, normalize=False)

In [ ]:
Audio(percussive, rate=sr, normalize=False)

### ② From scratch

Compute the RMS curve, then count the frames below its average. The plot marks those frames in red.

In [ ]:
def framed_rms(x):
    """RMS of each frame, with the frame starts from section 1."""
    frames = x[starts[:, np.newaxis] + np.arange(frame_length)]
    return np.sqrt(np.mean(frames**2, axis=1))


def plot_low_energy(ax, times, energy, title):
    """Plot an RMS curve, its average, and the frames below the average."""
    low = energy < energy.mean()
    ax.plot(times, energy, color="gray")
    ax.scatter(times[low], energy[low], color="C3", s=10, label="below average")
    ax.axhline(energy.mean(), color="k", linestyle="--", label="average")
    ax.set_title(f"{title}: low-energy rate {np.mean(low):.2f}")
    ax.set_xlabel("Time (s)")


fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_low_energy(axes[0], frame_centres, framed_rms(sustained), "held tone")
plot_low_energy(axes[1], frame_centres, framed_rms(percussive), "hits")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

### ③ mirpy

`mir.lowenergy` is MIRtoolbox `mirlowenergy`. You can give it the sound (it computes the RMS curve itself) or an RMS curve.

Two options change the definition:

- `threshold`: a frame counts as low when it is below `threshold` × the average (MATLAB `'Threshold'`, default 1);
- `asr=True`: the **Average Silence Ratio** (Feng, Zhuang, Pan, 2003), which uses the RMS without the square root and a threshold of 0.5 (MATLAB `'ASR'`).

In [ ]:
for name, x in [("held tone", sustained), ("hits", percussive)]:
    sound = mir.audio(x, sr=sr)
    print(
        f"{name:>9}: rate {mir.lowenergy(sound).data.item():.2f}, "
        f"threshold 0.9 {mir.lowenergy(sound, threshold=0.9).data.item():.2f}, "
        f"ASR {mir.lowenergy(sound, asr=True).data.item():.2f}"
    )

With `threshold=0.9` fewer frames of the held tone count as low, since they all sit close to the average. The hits do not change: their quiet frames are far below the average.

### ④ librosa and ⑤ Essentia

Neither library has a low-energy function. It takes one line on top of their RMS curves. Here they are with the same frames as mirpy, starting at 0 (`center=False` in librosa).

In [ ]:
librosa_energy = librosa.feature.rms(
    y=percussive, frame_length=frame_length, hop_length=int(hop), center=False
)[0]
print(f"librosa:  {np.mean(librosa_energy < librosa_energy.mean()):.2f}")

if es is not None:
    essentia_energy = essentia_rms_curve(percussive)
    print(f"Essentia: {np.mean(essentia_energy < essentia_energy.mean()):.2f}")

### ⑥ Side by side

All three give the same answer for the hits: most frames are below the average. The rate counts frames, so it depends on the frame settings. Always report them with the value.

### A real recording

An excerpt of Brahms's Hungarian Dance No. 5, played by the US Army Strings (public domain, from [Musopen](https://musopen.org/music/43805-hungarian-dance-no-5-in-f-sharp-minor-woo-1-string-orch); librosa downloads it the first time).

In [ ]:
brahms = mir.audio(librosa.ex("brahms"))
Audio(brahms.data[0, :, 0], rate=brahms.sr)

In [ ]:
brahms_rms = mir.rms(brahms, frame=True)

fig, ax = plt.subplots(figsize=(10, 3))
plot_low_energy(ax, brahms_rms.frame_centers, brahms_rms.data[0, 0], "Brahms")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.show()

The loud opening and the loud passage near 37 s are above the average; the softer middle passages and the fade at the end are below it. In this excerpt the two balance out, and the rate is close to 0.5.

### Exercise

Compute the low-energy rate of the Brahms excerpt with frames of 20 ms, 50 ms, 200 ms and 1 s (hop: half a frame). Plot the rate against the frame length. Why does it change?

In [ ]:
# Your answer here.

Solution (click to show):

In [ ]:
lengths = [0.02, 0.05, 0.2, 1.0]
rates = [mir.lowenergy(brahms, frame=(length, 0.5)).data.item() for length in lengths]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(lengths, rates, marker="o")
ax.set_xscale("log")
ax.set_xticks(lengths, ["20 ms", "50 ms", "200 ms", "1 s"])
ax.minorticks_off()
ax.set_xlabel("Frame length")
ax.set_ylabel("Low-energy rate")
plt.show()

The rate falls as the frames get longer. Short frames catch brief peaks, such as the start of a bow stroke. These peaks raise the average, so more frames fall below it. Long frames smooth the peaks away, and fewer frames are below the average.

## Summary

| MIRtoolbox | mirpy | librosa | Essentia |
|---|---|---|---|
| `mirframe(a, 0.05, 0.5)` | `mir.frame(a, (0.05, 0.5))` | `librosa.util.frame` | `FrameGenerator` |
| `mirrms(a)` | `mir.rms(a)` | `np.sqrt(np.mean(y**2))` | `RMS()(y)` |
| `mirrms(a, 'Frame')` | `mir.rms(a, frame=True)` | `librosa.feature.rms` | `FrameGenerator` + `RMS` |
| `mirlowenergy(a)` | `mir.lowenergy(a)` | from `feature.rms` | from `RMS` |

## Next

C4a Spectral shape looks at the frequencies inside each frame: how bright or noisy a sound is.

## References

As cited in the MIRtoolbox 1.8.1 User's Manual.

- Lartillot, O., Toiviainen, P. "A Matlab Toolbox for Musical Feature Extraction From Audio", International Conference on Digital Audio Effects, Bordeaux, 2007. *MIRtoolbox.*
- Tzanetakis, G., Cook, P. Musical genre classification of audio signals. IEEE Tr. Speech and Audio Processing, 10(5), 293–302, 2002. *Low-energy rate.*
- Feng, Y., Zhuang, Y., Pan, Y. (2003) Popular music retrieval by detecting mood, ACM SIGIR Conference on Research and Development in Information Retrieval. *Average Silence Ratio (`asr=True`).*